# 04 — Multimodal calibration (RS-SPSO)

Respawning speciation-based PSO searches for multiple distinct GR4J parameter sets that all calibrate well — the basis of the equifinality analysis. Same KGE objective and 730-day warm-up as notebook 03.

In [1]:
import import_ipynb
from Metric_Calculation import compute_KGE, compute_PBIAS, compute_NSE
from GR4J import compute_Q
from rs_spso import rs_spso
import pandas as pd
import numpy as np


2.56 ms ± 183 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
5.684341886080802e-14
[0.67712195 0.62997907 0.58832275 ... 0.60016149 0.58716065 0.92408118]


In [2]:
csv_path = r"D:\Claude\flood_hydrology_modeling\data\processed\prec_PET_sf.csv"
out_path = r"D:\Claude\flood_hydrology_modeling\data\processed\GR4J.csv"
df = pd.read_csv(csv_path)
P = df['prec_AGCD']
E = df['PET_morton']
obs = df['sf_mmd'].to_numpy(dtype=np.float64)   # keep NaN as-is, don't fill
warmup = 730  # 2-year warm-up (2*365)
x1, x2, x3, x4 = [350, 0, 90, 1.7]
x = np.array((x1,x2,x3,x4))

In [3]:
# Step 2 - the scoring wrapper (warm-up slice + NaN mask, returns KGE):
def score_kge(sim, obs, warmup):
    s = sim[warmup:]
    o = obs[warmup:]
    mask = ~np.isnan(o)
    return compute_KGE(o[mask], s[mask])[0]     # [0] = the KGE scalar

# Step 3 - the objective. Search the NORMALIZED cube [0,1]^4 so RS-SPSO's species
# distance treats all four GR4J parameters fairly - raw X1 (1-1500) and X3 (1-500)
# would otherwise dominate the Euclidean distance and hide X2/X4. Denormalize z -> x:

LOWER = np.array([1.0, -5.0, 1.0, 0.5])        # x1, x2, x3, x4 lower bounds
UPPER = np.array([1500.0, 5.0, 500.0, 4.0])    # x1, x2, x3, x4 upper bounds

def denormalize(z):
    return LOWER + np.asarray(z) * (UPPER - LOWER)

def objective(z):
    x = denormalize(z)           # [0,1]^4 -> physical GR4J parameters
    sim = compute_Q(P, E, x)
    kge = score_kge(sim, obs, warmup)
    if not np.isfinite(kge):
        return 1e6               # push optimizer away from bad params
    return -kge

# Step 4 - run RS-SPSO over the unit cube:

sols = rs_spso(objective, [(0, 1)] * 4, m=4, N=60, max_iteration=300)
print(f"found {len(sols)} distinct GR4J parameter set(s):")
for pos, score in sols:
    x = denormalize(pos)         # report physical parameters, not [0,1] coords
    print(f"  x1={x[0]:8.2f}  x2={x[1]:+6.3f}  x3={x[2]:8.2f}  x4={x[3]:6.3f}   KGE={-score:.4f}")


[iter  15] species 1 stalled -> DE refine  f=-8.350e-01  (9.72s)
[iter  15] species 1 -> NEW optimum (1/4)  f=-8.350e-01
[iter  16] species 0 stalled -> DE refine  f=-8.698e-01  (7.52s)
[iter  16] species 0 -> NEW optimum (2/4)  f=-8.698e-01
[iter  27] species 3 stalled -> DE refine  f=-8.618e-01  (8.24s)
[iter  27] species 3 -> NEW optimum (3/4)  f=-8.618e-01
[iter  31] species 2 stalled -> DE refine  f=-8.742e-01  (9.28s)
[iter  31] species 2 -> NEW optimum (4/4)  f=-8.742e-01
RS-SPSO: 4/4 solutions in 40.95s
found 4 distinct GR4J parameter set(s):
  x1=  213.42  x2=+4.374  x3=  271.96  x4= 0.675   KGE=0.8350
  x1=   20.96  x2=-3.033  x3=  265.03  x4= 1.023   KGE=0.8698
  x1=   15.09  x2=-4.500  x3=  285.01  x4= 0.675   KGE=0.8618
  x1=   59.92  x2=+0.249  x3=  275.35  x4= 1.013   KGE=0.8742


In [17]:
def score_PBIAS(sim, obs, warmup):
    s = sim[:warmup]
    o = obs[:warmup]
    mask = ~np.isnan(o)
    return compute_KGE(o[mask], s[mask])[0] 
for pos, score in sols:
    x = denormalize(pos)         # report physical parameters, not [0,1] coords
    sim = compute_Q(P, E, x)
    pBIAS = score_PBIAS(sim, obs, warmup)
    print(pBIAS)

0.8995598711065669
0.8220236848794326
0.799479646785175
0.8458978188255983
